# 作业2：运力调度（完成版）

保留教师 Notebook 的动态环境、TBG、BB、NSGA-II 四部分。原件不变；生产实现集中在 `task2/src/online.py`，这里逐段导入、运行与核验，避免 Notebook 与脚本维护两套算法。TODO1—5、finalize 和 Assignment 生成均有实际实现与反例。

课件场景是 10×10 km、20 min、20 对象、1 km/min；原 Notebook 的100对象、1000空间单位/2000时间单位另立示例。签到是代理需求，不是真实网约车单。

In [1]:
from pathlib import Path
import sys, os, json
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['OMP_NUM_THREADS']='1'
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'task2/src/online.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from task2.scripts.build_figures import P, COLORS
np.set_printoptions(precision=4,suppress=True)

ImportError: cannot import name 'backend2gui' from 'IPython.core.pylabtools' (/home/addaswsw/lab/Smart_Cities_and_Location_Services/.venv/lib/python3.12/site-packages/IPython/core/pylabtools.py)

## 1. 双边在线事件与两种离线目标

每个司机/订单只匹配一次；同刻先请求后司机（各按ID），同刻到达全部处理完再开批次。延迟=等待+距离/速度。离线分别求最小总延迟与最小最大延迟；它们是两个不同问题。

In [ ]:
from task2.src.online import (Worker,Request,Assignment,SimulationEnvironment,TBGAlgorithm,BBAlgorithm,
    run_simulation,offline_reference,summarize,instance_ratios,bottleneck_assignment)
from scipy.optimize import linear_sum_assignment
C=np.array([[0.,6.],[6.,10.]])
i,j=linear_sum_assignment(C);bi,bj=bottleneck_assignment(C)
assert C[i,j].sum()==10 and C[i,j].max()==10
assert C[bi,bj].max()==6
print('sum-optimum max=10; bottleneck-optimum max=6（精确反例）')
sim=SimulationEnvironment(123)
sim.generate_synthetic_data(20,20,(0,20),(0,10))
opt_sum=offline_reference(sim.workers,sim.requests,objective='sum')
opt_max=offline_reference(sim.workers,sim.requests,objective='max')

## 2. TBG：完成原 Cell4 的TODO1—5与finalize

请求到达时取 travel≤gamma 的最近司机；司机到达优先最早超阈值请求，否则取总延迟<gamma的最小者。gamma的单位是时间，距离要除以速度。finalize只使用真实剩余司机，不能制造供给。小gamma不会保证更低最大延迟。

In [ ]:
rows=[]
for gamma in [.5,1,2,5,10,15]:
    method=TBGAlgorithm(gamma,speed=1)
    a=run_simulation(sim.workers,sim.requests,method,20)
    rows.append(dict(gamma=gamma,**summarize(a,20,20),**instance_ratios(a,opt_sum,opt_max)))
display(pd.DataFrame(rows).round(4))
assert len(run_simulation([],sim.requests,TBGAlgorithm(5),20))==0

## 3. BB：原 Cell8 的Assignment已给出，真正风险是丢弃队列

sum-KM是Notebook定义；课件的瓶颈版本另标BBmax。两者均保留未匹配对象再尝试。丢弃策略只作有明确身份的对照。下表同时给出覆盖率，部分匹配不计算与全服务离线最优的实例比值。

In [ ]:
rows=[]
for theta in [.5,1,2,5]:
    for recycle,objective in [(True,'sum'),(True,'max'),(False,'sum')]:
        a=run_simulation(sim.workers,sim.requests,BBAlgorithm(theta,recycle=recycle,objective=objective),20)
        rows.append(dict(theta=theta,recycle=recycle,objective=objective,
            **summarize(a,20,20),**instance_ratios(a,opt_sum,opt_max)))
display(pd.DataFrame(rows).round(4))

### Notebook原尺度的theta=50/100/200

这里速度=1空间单位/时间单位，绝不把数值1000直接称作km。该场景独立于上一课件场景。

In [ ]:
nb_sim=SimulationEnvironment(123)
nb_sim.generate_synthetic_data(100,100,(0,2000),(0,1000))
rows=[]
for theta in [50,100,200]:
    a=run_simulation(nb_sim.workers,nb_sim.requests,BBAlgorithm(theta),2000)
    rows.append(dict(theta=theta,**summarize(a,100,100)))
display(pd.DataFrame(rows).round(4))

## 4. 基础NSGA-II：接驾距离与送客距离

独立合成20司机/20个OD，10×10km，收入目标=欧氏送客km。没有套用进阶的5+2d车费。五seed、种群100、300代。静态多目标前沿需先给标量偏好，才有同目标的精确比较；这里是实例近似比，不是在线理论竞争保证。

In [ ]:
from task2.src.evolution import DispatchProblem,run_nsga,weighted_km
rng=np.random.default_rng(2024)
drv,origin,dest=(rng.uniform(0,10,(20,2)) for _ in range(3))
pickup=np.linalg.norm(drv[:,None]-origin[None],axis=-1)
trip=np.linalg.norm(dest-origin,axis=1)
problem=DispatchProblem(dict(drv=drv,o_xy=origin,d_xy=dest,pickup=pickup,fare=trip),K=20,radius=np.sqrt(200))
basic_runs=[run_nsga(problem,pop_size=100,n_gen=300,seed=s) for s in range(5)]
fig,ax=plt.subplots(figsize=(7,4))
for s,r in enumerate(basic_runs):ax.scatter(r['F'][:,1],-r['F'][:,0],s=10,label=f'seed{s}',alpha=.6)
ax.set(xlabel='Pickup (km)',ylabel='Delivered OD distance (km)');ax.legend();plt.show()
rows=[]
for reference in weighted_km(problem,[0,.25,.5,1,2]):
    lam=reference['lam'];opt=-reference['F'][0]-lam*reference['F'][1]
    for s,r in enumerate(basic_runs):
        val=np.max(-r['F'][:,0]-lam*r['F'][:,1])
        rows.append(dict(seed=s,lam=lam,optimal_utility=opt,obtained_utility=val,
                         instance_ratio=opt/val if val>0 else np.nan))
display(pd.DataFrame(rows).round(5))

## 5. 真实签到代理的空间需求扩展

使用前段训练日期的签到热点和相邻签到OD起点。将来源相同小时的起点有放回采样成60min模拟流，司机到达为模拟随机数；没有把不同日期的签到拼成真实出租车运营日。当地时间口径中的12对UTC矛盾有单独稳健性记录。

In [ ]:
from task2.src.dispatch_data import load_data
bundle=load_data()
display(pd.DataFrame(bundle.audit['split']).T)
rng=np.random.default_rng(10)
pool=bundle.trips_train_inner[bundle.trips_train_inner.minute//60==18]
od=pool.iloc[rng.choice(len(pool),20)]
hot=bundle.checkins_train_inner.iloc[rng.choice(len(bundle.checkins_train_inner),20)]
w=[Worker(i,(float(row.x),float(row.y)),float(rng.uniform(0,60))) for i,(_,row) in enumerate(hot.iterrows())]
r=[Request(i,(float(row.ox),float(row.oy)),float(row.minute-1080)) for i,(_,row) in enumerate(od.iterrows())]
display(pd.DataFrame([dict(method=name,**summarize(run_simulation(w,r,algo,60),20,20)) for name,algo in
    [('TBG5',TBGAlgorithm(5)),('BBsum2',BBAlgorithm(2)),('BBmax2',BBAlgorithm(2,objective='max'))]]).round(4))

完整55个开发实例、独立合成确认、单因素阈值、4时段代理实验及逐匹配记录分别由`run_online.py`与`run_online_studies.py`生成；本Notebook所有表和图来自当前Kernel实际执行。正式大批次结果与报告通过`docs/REPRODUCIBILITY_GUIDE.md`定位。